# HiGAN+ Recognizer-Crop Experiment: `char_aligned`

Per-sample uniform random char-aligned slice [i:j] with length >= 1. Applied with prob=0.5 per iter. The most aggressive crop: tests how much R-pressure can be removed before content quality collapses.

**This notebook trains exactly one experiment.**  Three sister notebooks
under `docs/` cover the other variants:
`kaggle_train_baseline.ipynb`, `kaggle_train_left_half.ipynb`,
`kaggle_train_left_3q.ipynb`, `kaggle_train_char_aligned.ipynb`.

Run them in separate Kaggle sessions (ideally separate accounts) so each
experiment gets its own 9-hour budget without competing for GPU.

**Settings before pressing Run:**
- *Settings* → Accelerator: **GPU T4 / P100** and **Internet: ON**.
- *Add-ons* → Secrets → toggle **WANDB_API_KEY** on (optional but strongly recommended for cross-experiment comparison).
- *Persistence*: keep the default `Files only` so checkpoints in `/kaggle/working/` survive after the session ends.

Config used: `configs/gan_iam_crop_char_aligned.yml`.
All output lands under `runs/gan_iam_crop_char_aligned-<m-d-H-M>/`.


## 1. Environment check

In [ ]:
import torch, sys
print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__)
print('cuda    :', torch.version.cuda)
print('gpu     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only!')


## 2. Clone repo (branch `feat/recog-random-crop`)

Idempotent: re-running this cell after a kernel restart only fast-forwards the existing clone.

In [ ]:
%cd /kaggle/working
BRANCH = 'feat/recog-random-crop'
![ -d higanplus ] || git clone --depth 1 -b $BRANCH https://github.com/ks2n/higanplus.git
%cd /kaggle/working/higanplus
!git fetch origin $BRANCH --depth=1 && git checkout $BRANCH && git pull --ff-only origin $BRANCH
!git log -1 --oneline


## 3. Install Python deps

Kaggle's image already ships CUDA-built PyTorch, so we skip torch and only install the small pure-Python helpers from `requirements.txt` (now includes `wandb`).

In [ ]:
!pip install -q -r requirements.txt


## 4. Download IAM dataset + 4 pretrained checkpoints

Fetches the upstream-released hdf5 files and the four `.pth` checkpoints (`deploy_HiGAN+`, `ocr_iam_new`, `wid_iam_new`, `wid_iam_test`).  ~1 GB total, ~3-5 minutes on Kaggle's outbound link.  Idempotent: skips files already on disk.

In [ ]:
!bash scripts/setup_data.sh


## 5. wandb login (optional but recommended)

Kaggle stores secrets in `kaggle_secrets.UserSecretsClient`, **not** `os.environ`.  This cell tries the Kaggle path first, falls back to env-var when running locally, and exports the key back to the env so the `train.py` subprocess inherits it.

In [ ]:
import os, subprocess, sys

key = None
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    key = os.environ.get('WANDB_API_KEY')

if not key:
    print('[wandb] WANDB_API_KEY not found; this run will only log to TensorBoard.')
    print('         Settings -> Secrets -> Add WANDB_API_KEY, then toggle it on for this notebook.')
else:
    os.environ['WANDB_API_KEY'] = key
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'])
    subprocess.check_call(['wandb', 'login', '--relogin', key])
    print('[wandb] login OK; key exported to env for child processes.')


## 6. Smoke test (5 iters x 2 epochs)

Before burning hours on the full experiment, run the smoke config once to
verify:

- Kaggle's GPU + the cloned repo + the downloaded data can talk to each other.
- The new `recog_crop` code path (Idea 1) actually executes without crashing
  -- the smoke uses `mode=char_aligned, prob=1.0` so every iter hits the
  cropped CTC pathway.

Expect ~3-4 minutes on T4.  CTC-fake will read ~25-30 because G is random init.

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_crop_smoke.yml


## 7. Train (full 70 epochs, FROM SCRATCH for G/D/E)

This kernel does **not** load `deploy_HiGAN+.pth`.  Only the auxiliary teachers
(R, W, B) are warm-started from the author's released checkpoints.  Generator,
discriminators, and style encoder all start from random init -- the same setup
as the upstream paper's `gan_iam.yml`.  This is required for a fair comparison
with the three crop variants: any FID/KID improvement we measure must come from
the crop trick changing **how G learns**, not from G already being trained.

Logs:
- TensorBoard: `runs/<config-name>-<m-d-H-M>/`
- wandb: project `higanplus-recog-crop`, group `recog-crop-v1`

**Time budget.** A from-scratch HiGAN+ run at batch=4 on T4 is ~30-35 hours
(LR linear decay starts at epoch 25, ends at epoch 70).  Kaggle sessions are
capped at 9 hours, so plan on **4-5 sessions per experiment**.  Workflow each
session:

1. Run cells 1-5 (clone, deps, data, wandb).
2. Run cell 8 (resume helper) -- it patches the config to load `last.pth` from
   the previous session if any exists, otherwise it's a no-op for the first
   session.
3. Run this cell.
4. When the session is about to expire, stop the cell and save the notebook.

Sample images written to `runs/<...>/imgs/` every 500 G-steps.  Don't expect
clean handwriting before ~epoch 5-10.

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_crop_char_aligned.yml


## 8. Resume helper (scoped to this experiment only)

Looks for the latest `runs/gan_iam_crop_char_aligned-*` directory and patches
`configs/gan_iam_crop_char_aligned.yml` so the next `train.py` call starts from
that run's `last.pth`.  This is intentionally narrow: it will **not**
touch the other three experiments' configs even if their runs are newer.

In [ ]:
import glob, os, re

RUN_PREFIX = 'gan_iam_crop_char_aligned'
CFG_PATH = f'/kaggle/working/higanplus/HiGAN+/configs/gan_iam_crop_char_aligned.yml'

candidates = sorted(
    glob.glob(f'/kaggle/working/higanplus/HiGAN+/runs/{RUN_PREFIX}-*'),
    key=os.path.getmtime,
)
if not candidates:
    print(f'no {RUN_PREFIX} runs found yet -- nothing to resume from')
else:
    run_dir = candidates[-1]
    last_ckpt = os.path.join(run_dir, 'ckpts', 'last.pth')
    print('latest run :', run_dir)
    print('latest ckpt:', last_ckpt, '(exists:', os.path.exists(last_ckpt), ')')
    with open(CFG_PATH) as f: txt = f.read()
    new_txt = re.sub(r"pretrained_ckpt:\s*'[^']*'", f"pretrained_ckpt: '{last_ckpt}'", txt)
    if new_txt != txt:
        with open(CFG_PATH, 'w') as f: f.write(new_txt)
        print(f'patched {CFG_PATH} -> resume on next train.py call')
    else:
        print('config already references the latest ckpt; nothing to do')


## 9. Inspect outputs (this experiment only)

Disk usage of every `gan_iam_crop_char_aligned-*` run plus the latest sample image generated.

In [ ]:
import glob, os

RUN_PREFIX = 'gan_iam_crop_char_aligned'
!du -sh /kaggle/working/higanplus/HiGAN+/runs/gan_iam_crop_char_aligned-* 2>/dev/null || echo 'no runs yet'
for run in sorted(glob.glob(f'/kaggle/working/higanplus/HiGAN+/runs/{RUN_PREFIX}-*')):
    imgs = sorted(glob.glob(os.path.join(run, 'imgs', '*.png')))
    if imgs:
        print(run, '->', os.path.basename(imgs[-1]))
